# Fig 1 

In [ ]:
DoDimPlot <- function(object,vars,clusters,mycolor,label.size,title = NULL,legend = NULL){
        data <- data.frame(x = 0,y =0)
        p.arrow <- ggplot(data = data,aes(x = x,y = y)) + geom_point(size=2, alpha=1) + 
            geom_segment(aes(,x = x , y = y , xend = x +4, yend = y ), colour = "black",
                     size=0.5,arrow = arrow(length = unit(0.5,"cm")))+ 
            geom_segment(aes(x = x  , y = y , xend = x , yend =y + 4), colour = "black", 
                     size=0.5,arrow = arrow(length = unit(0.5,"cm"))) +
            annotate("text", x = data$x+2, y =data$y-1, label = "tSNE_1", color="black",
                 size = 4,fontface="bold" ) + 
            annotate("text", x = data$x-1, y =data$y + 2, label = "tSNE_2",color="black",
                 size = 4, fontface="bold" ,angle=90) + theme_bw()+
            theme(plot.margin=unit(c(0.1, 0.1, 0.1, 0.1), "inches"),legend.title=element_blank()) +
                theme(panel.grid.major = element_blank(), #主网格线
                panel.grid.minor = element_blank(), #次网格线
                panel.border = element_blank(), #边框
                axis.title = element_blank(),  #轴标题
                axis.text = element_blank(), # 文本
                axis.ticks = element_blank())
        # mycolor <- colorRampPalette(brewer.pal(9, "Set1"))(20)
        plot.data <- FetchData(object,vars = vars)
        plot.data$clusters <- as.character(plot.data$clusters) # !!!
        plot.data <- plot.data %>% mutate(celltype2 = paste(clusters,celltypes,sep = " "))
        plot.data$celltype2 <- factor(plot.data$celltype2,levels = clusters$celltype)
        print(unique(plot.data$celltype2))
        centers <- plot.data %>% group_by(clusters) %>% summarise(tSNE_1 = median(tSNE_1),tSNE_2 = median(tSNE_2)) %>%
                as.data.frame()  ### ???? 
        print(head(plot.data))
        print(head(centers))
        theme_self <- theme(plot.margin=unit(c(0.1, 0.1, 0.1, 0.1), "inches"),legend.title=element_blank()) +
                   theme(panel.grid.major = element_blank(), #主网格线
                   panel.grid.minor = element_blank(), #次网格线
                   panel.border = element_blank(), #边框
                   axis.title = element_blank(),  #轴标题
                   axis.text = element_blank(), # 文本
                   axis.ticks = element_blank(),plot.title = element_text(size =25,hjust = 0.5,
                                                                          family = "Times",face = "bold"))
        if(is.null(legend)){
            theme_self <-theme_self + theme(legend.position = "none")
        } else {
            theme_self <- theme_self + theme(legend.text=element_text(colour="black",size=12),legend.position="right",
                          plot.title=element_text(hjust=0.5))
        }
        # stop("haha")
        p <- ggplot(data=plot.data,aes(x=tSNE_1, y=tSNE_2))+
            geom_point_rast(alpha=0.2, size=0.0000000001,aes(colour=celltype2)) +
            guides(colour=guide_legend(override.aes=list(size=5,alpha = 1)))+
            scale_color_manual(values=mycolor)+
            theme_bw()+
            labs(x= "tSNE_1",y="tSNE_2",title= title) + theme_self +  
             geom_point(data=centers, mapping=aes(x=tSNE_1, y=tSNE_2), size=0, alpha=0) + 
             geom_text(data=centers, mapping=aes(label=clusters), size=label.size, fontface="plain")
       # right 0.15 top 0.2 # w = 10, h = 8 #
       # right 0.2 top 0.2 # w = 8, h = 8 #
        p <- p + inset_element(p.arrow,left = 0,bottom = 0,right = 0.2,top = 0.2,align_to= "full",on_top = TRUE)
        return(p)                                                        
}

In [ ]:
GetMergeTable <- function(Object,var){
    print(Object)
    whole <- as.data.frame.array(table(Object@meta.data$celltypes,Object@meta.data$mice))
    print(whole)
    Idents(Object) <- var
    for( i in unique(Object@meta.data[,var])){
        print(i)
        subObj <- subset(Object, idents = i)
        tissue_table <- as.data.frame.array(table(subObj@meta.data$celltypes,subObj@meta.data$mice))
        tissue_prefix <- rep(i,2)
        table_newnames <- paste(colnames(tissue_table),tissue_prefix,sep = "_")
        colnames(tissue_table) <- table_newnames
        whole <- cbind(whole,tissue_table)
    }
    return(whole)
}
CellRadioNumPlot <- function(Object,var,tissue_levels,mycolor = NULL,levels = NULL){
    CellNum2 <- GetMergeTable(Object,var)
    CellNum2 <- CellNum2[,3:18]
    # Radio #
    CellNum.p <- as.data.frame(proportions(as.matrix(CellNum2),margin = 2) * 100)
    CellNum.p$celltypes <- rownames(CellNum.p)
    CellNum.pt <- melt(CellNum.p,id.vars= "celltypes")
    colnames(CellNum.pt) <- c("celltypes","variable","Composition")
    print(head(CellNum.pt))
    # cell num #
    rownames(CellNum2) <- NULL
    Tissue_CellNum <- as.data.frame(colSums(CellNum2))
    Tissue_CellNum$tissue <- rownames(Tissue_CellNum)
    print(Tissue_CellNum)
    colnames(Tissue_CellNum) <- c('num','tissue')
    if(!is.null(levels)){
            CellNum.pt$celltypes <- factor(CellNum.pt$celltypes,levels = levels)  # celltype  levels color 
    } else {
        print('No level!')
    }
    CellNum.pt$variable <- factor(CellNum.pt$variable,levels = tissue_levels) # tissue levels x axis #
    Tissue_CellNum$tissue <- factor(Tissue_CellNum$tissue,levels = tissue_levels)
    if(is.null(mycolor)){
        nclu <- length(rownames(CellNum.p))
        mycolor = colorRampPalette(brewer.pal(9, "Set1"))(nclu)
    } else {
        print(mycolor)
     }
    
    # Radio #
    p1 <- ggplot(CellNum.pt,aes(x = variable,y = Composition,fill= celltypes,stratum = celltypes,alluvium = celltypes)) + 
            geom_col(width = 0.5,color= "black") +  coord_flip() +
            geom_flow(width = 0.5,alpha = 0.4, knot.pos=0.5) + theme_classic() + labs(x = " " , y = "Composition (%)") +  
    # scale_fill_manual(values=mycolor) + theme(
    #        axis.text.y = element_text(size = 20,family = "Times",face = "bold",margin  = margin(0,10,0,0)),
    #        axis.text.x = element_text(size = 20,family = "Times",face = "bold",margin  = margin(10,0,0,0)),
    #        axis.title.x= element_text(size = 30,family = "Times",face = "bold",margin  = margin(15,0,0,0)),
    #        axis.title.y= element_text(size = 30,family = "Times",face = "bold",margin  = margin(0,15,0,0)),
    #         axis.line = element_line(linetype = 1,color= "black",size = 1),
    #        legend.text = element_text(size = 18,family = "Times",face = "bold"),
    #         legend.title = element_text(size = 25,family = "Times",face = "bold"),
    # axis.ticks  = element_line(color = "black",size = 1,lineend = 2))
    scale_fill_manual(values=mycolor) + theme(
            axis.text.y = element_text(size = 20,face = "bold",margin  = margin(0,10,0,0)),
            axis.text.x = element_text(size = 20,face = "bold",margin  = margin(10,0,0,0)),
            axis.title.x= element_text(size = 30,face = "bold",margin  = margin(15,0,0,0)),
            axis.title.y= element_text(size = 30,face = "bold",margin  = margin(0,15,0,0)),
             axis.line = element_line(linetype = 1,color= "black",size = 1),
            legend.text = element_text(size = 18,face = "bold"),
             legend.title = element_text(size = 25,face = "bold"),
           axis.ticks  = element_line(color = "black",size = 1,lineend = 2))
    # cell num #
    p2 <- ggplot(data2,aes(x = tissue, y = num)) +
                geom_bar(stat = "identity",position = "stack",width = 0.5) +
                theme_classic() + labs(x = "" , y = "Cell Number") +
                scale_fill_manual(values=mycolor) + theme(
                    axis.text.y = element_blank(),
                    axis.text.x = element_text(size = 20,face = "bold", # family = "Times"
                                           margin = margin(10,0,0,0)),
                    axis.title.x = element_text(margin  = margin(15,0,0,0)),
                    axis.title.y = element_text(margin  = margin(0,15,0,0)),
                    axis.title= element_text(size = 30,face = "bold"), # ,family = "Times"
                    axis.line = element_line(linetype = 1,color= "black",size = 1),
                    legend.text = element_text(size = 18 ,face = "bold"), # ,family = "Times"
                    legend.title = element_text(size = 25,face = "bold"),# family = "Times"
                    axis.ticks.y = element_blank(),
                    axis.ticks.x = element_line(color = "black",size = 1,lineend = 2)) + coord_flip() +
                            scale_y_continuous(breaks = c(0,10000,20000,30000),
                           labels = c("0","10k","20k","30k"))
    p.me <- p1 + p2 + plot_layout(width = c(3,1),guides = 'collect')
    return(p.me)
} 


In [ ]:
def get_rgb_function(cmap, min_value, max_value):
    r"""Generate a function to map continous values to RGB values using colormap between min_value & max_value."""

    if min_value > max_value:
        raise ValueError("Max_value should be greater or than min_value.")

    if min_value == max_value:
        warnings.warn(
            "Max_color is equal to min_color. It might be because of the data or bad parameter choice. "
            "If you are using plot_contours function try increasing max_color_quantile parameter and"
            "removing cell types with all zero values."
        )

        def func_equal(x):
            factor = 0 if max_value == 0 else 0.5
            return cmap(np.ones_like(x) * factor)

        return func_equal

    def func(x):
        return cmap((np.clip(x, min_value, max_value) - min_value) / (max_value - min_value))

    return func


def rgb_to_ryb(rgb):
    """
    Converts colours from RGB colorspace to RYB

    Parameters
    ----------

    rgb
        numpy array Nx3

    Returns
    -------
    Numpy array Nx3
    """
    rgb = np.array(rgb)
    if len(rgb.shape) == 1:
        rgb = rgb[np.newaxis, :]

    white = rgb.min(axis=1)
    black = (1 - rgb).min(axis=1)
    rgb = rgb - white[:, np.newaxis]

    yellow = rgb[:, :2].min(axis=1)
    ryb = np.zeros_like(rgb)
    ryb[:, 0] = rgb[:, 0] - yellow
    ryb[:, 1] = (yellow + rgb[:, 1]) / 2
    ryb[:, 2] = (rgb[:, 2] + rgb[:, 1] - yellow) / 2

    mask = ~(ryb == 0).all(axis=1)
    if mask.any():
        norm = ryb[mask].max(axis=1) / rgb[mask].max(axis=1)
        ryb[mask] = ryb[mask] / norm[:, np.newaxis]

    return ryb + black[:, np.newaxis]


def ryb_to_rgb(ryb):
    """
    Converts colours from RYB colorspace to RGB

    Parameters
    ----------

    ryb
        numpy array Nx3

    Returns
    -------
    Numpy array Nx3
    """
    ryb = np.array(ryb)
    if len(ryb.shape) == 1:
        ryb = ryb[np.newaxis, :]

    black = ryb.min(axis=1)
    white = (1 - ryb).min(axis=1)
    ryb = ryb - black[:, np.newaxis]

    green = ryb[:, 1:].min(axis=1)
    rgb = np.zeros_like(ryb)
    rgb[:, 0] = ryb[:, 0] + ryb[:, 1] - green
    rgb[:, 1] = green + ryb[:, 1]
    rgb[:, 2] = (ryb[:, 2] - green) * 2

    mask = ~(ryb == 0).all(axis=1)
    if mask.any():
        norm = rgb[mask].max(axis=1) / ryb[mask].max(axis=1)
        rgb[mask] = rgb[mask] / norm[:, np.newaxis]

    return rgb + white[:, np.newaxis]

# add function of gradient custom #  2024 - 1 - 10 # lwm 
def gradient(start_color,end_color,num_steps):
    '''
    Generates a list of gradient colors between two colors
    :parameter 
         start_color : The RGB value of the starting color
         end_color : The RGB value of the ending color
         num_steps : num of gradient colors 
    :return 
        gradient_colors : list of gradient colors
    '''
    r = np.linspace(start_color[0],end_color[0],num_steps)/255
    g = np.linspace(start_color[1],end_color[1],num_steps)/255
    b = np.linspace(start_color[2],end_color[2],num_steps)/255
    # value of RGB to lsit of 
    gradient_colors = np.array([[r[i],g[i],b[i],1]for i in range(num_steps)])
    gradient_colors[:,3] = np.repeat(0.7,num_steps)
    #print("gradient_colors:",gradient_colors)
    
    return gradient_colors


# plot # 
def plot_spatial_general1(
    value_df,
    coords,
    labels,
    min_intensity = None, # accept number ! 2023-8-31
    max_intensity = None,
    text=None,
    circle_diameter=4.0,
    alpha_scaling=1.0,
    max_col=(np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf),
    max_color_quantile=0.98,
    show_img=True,
    img=None,
    img_alpha=1.0,
    adjust_text=False,
    plt_axis="off",
    axis_y_flipped=True,
    x_y_labels=("", ""),
    crop_x=None,
    crop_y=None,
    text_box_alpha=0.9,
    reorder_cmap=range(7),
    style="fast",
    colorbar_position="bottom",
    colorbar_label_kw={},
    colorbar_shape={},
    colorbar_tick_size=12,
    colorbar_grid=None,
    image_cmap="Greys_r",
    white_spacing=20,
): 
    r"""Plot spatial abundance of cell types (regulatory programmes) with colour gradient and interpolation.

      This method supports only 7 cell types with these colours (in order, which can be changed using reorder_cmap).
      'yellow' 'orange' 'blue' 'green' 'purple' 'grey' 'white'

    :param value_df: pd.DataFrame - with cell abundance or other features (only 7 allowed, columns) across locations (rows)
    :param coords: np.ndarray - x and y coordinates (in columns) to be used for ploting spots
    :param text: pd.DataFrame - with x, y coordinates, text to be printed
    :param circle_diameter: diameter of circles
    :param labels: list of strings, labels of cell types
    :param 
    :param alpha_scaling: adjust color alpha
    :param max_col: crops the colorscale maximum value for each column in value_df.
    :param max_color_quantile: crops the colorscale at x quantile of the data.
    :param show_img: show image?
    :param img: numpy array representing a tissue image.
        If not provided a black background image is used.
    :param img_alpha: transparency of the image
    :param lim: x and y max limits on the plot. Minimum is always set to 0, if `lim` is None maximum
        is set to image height and width. If 'no_limit' then no limit is set.
    :param adjust_text: move text label to prevent overlap
    :param plt_axis: show axes?
    :param axis_y_flipped: flip y axis to match coordinates of the plotted image
    :param reorder_cmap: reorder colors to make sure you get the right color for each category

    :param style: plot style (matplolib.style.context):
        'fast' - white background & dark text;
        'dark_background' - black background & white text;

    :param colorbar_position: 'bottom', 'right' or None
    :param colorbar_label_kw: dict that will be forwarded to ax.set_label()
    :param colorbar_shape: dict {'vertical_gaps': 1.5, 'horizontal_gaps': 1.5,
                                    'width': 0.2, 'height': 0.2}, not obligatory to contain all params
    :param colorbar_tick_size: colorbar ticks label size
    :param colorbar_grid: tuple of colorbar grid (rows, columns)
    :param image_cmap: matplotlib colormap for grayscale image
    :param white_spacing: percent of colorbars to be hidden
    :param color_gradient : TURE of FALSE # lwm add 2024 - 1 - 10
    """
    
    if value_df.shape[1] > 7: # Default 7 : lwm : 9
        raise ValueError("Maximum of 7 cell types / factors can be plotted at the moment")

    def create_colormap(R, G, B):
        spacing = int(white_spacing * 2.55)

        N = 255 # others tsisue
        # N = 510  # spleen
        M = 3

        alphas = np.concatenate([[0] * spacing * M, np.linspace(0, 1.0, (N - spacing) * M)])

        vals = np.ones((N * M, 4))
        #         vals[:, 0] = np.linspace(1, R / 255, N * M)
        #         vals[:, 1] = np.linspace(1, G / 255, N * M)
        #         vals[:, 2] = np.linspace(1, B / 255, N * M)
        for i, color in enumerate([R, G, B]):
            vals[:, i] = color / 255
        vals[:, 3] = alphas

        return ListedColormap(vals)

    # Create linearly scaled colormaps ### add color ### 
    YellowCM = create_colormap(240, 228, 66)  # #F0E442 ['#F0E442', '#D55E00', '#56B4E9',
    # '#009E73', '#5A14A5', '#C8C8C8', '#323232']
    RedCM = create_colormap(213, 94, 0)  # #D55E00
    BlueCM = create_colormap(86, 180, 233)  # #56B4E9
    GreenCM = create_colormap(0, 158, 115)  # #009E73
    GreyCM = create_colormap(200, 200, 200)  # #C8C8C8
    WhiteCM = create_colormap(50, 50, 50)  # #323232
    PurpleCM = create_colormap(90, 20, 165)  # #5A14A5 
    GreenCM2 = create_colormap(139,251,122) 

    cmaps = [YellowCM, RedCM, BlueCM, GreenCM, PurpleCM,GreyCM, WhiteCM, GreenCM2]

    cmaps = [cmaps[i] for i in reorder_cmap]
    
 
    
    
    
    with mpl.style.context(style):
        fig = plt.figure()

        if colorbar_position == "right":
            if colorbar_grid is None:
                colorbar_grid = (len(labels), 1)

            shape = {"vertical_gaps": 1.5, "horizontal_gaps": 0, "width": 0.15, "height": 0.2}
            shape = {**shape, **colorbar_shape}

            gs = GridSpec(
                nrows=colorbar_grid[0] + 2,
                ncols=colorbar_grid[1] + 1,
                width_ratios=[1, *[shape["width"]] * colorbar_grid[1]],
                height_ratios=[1, *[shape["height"]] * colorbar_grid[0], 1],
                hspace=shape["vertical_gaps"],
                wspace=shape["horizontal_gaps"],
            )
            ax = fig.add_subplot(gs[:, 0], aspect="equal", rasterized=True)

        if colorbar_position == "bottom":
            if colorbar_grid is None:
                if len(labels) <= 3:
                    colorbar_grid = (1, len(labels))
                else:
                    n_rows = round(len(labels) / 3 + 0.5 - 1e-9)
                    colorbar_grid = (n_rows, 3)

            shape = {"vertical_gaps": 0.3, "horizontal_gaps": 0.6, "width": 0.2, "height": 0.035}
            shape = {**shape, **colorbar_shape}

            gs = GridSpec(
                nrows=colorbar_grid[0] + 1,
                ncols=colorbar_grid[1] + 2,
                width_ratios=[0.3, *[shape["width"]] * colorbar_grid[1], 0.3],
                height_ratios=[1, *[shape["height"]] * colorbar_grid[0]],
                hspace=shape["vertical_gaps"],
                wspace=shape["horizontal_gaps"],
            )

            ax = fig.add_subplot(gs[0, :], aspect="equal", rasterized=True)

        if colorbar_position is None:
            ax = fig.add_subplot(aspect="equal", rasterized=True)

        if colorbar_position is not None:
            cbar_axes = []
            for row in range(1, colorbar_grid[0] + 1):
                for column in range(1, colorbar_grid[1] + 1):
                    cbar_axes.append(fig.add_subplot(gs[row, column]))

            n_excess = colorbar_grid[0] * colorbar_grid[1] - len(labels)
            if n_excess > 0:
                for i in range(1, n_excess + 1):
                    cbar_axes[-i].set_visible(False)

        ax.set_xlabel(x_y_labels[0])
        ax.set_ylabel(x_y_labels[1])

        if img is not None and show_img:
            ax.imshow(img, aspect="equal", alpha=img_alpha, origin="lower", cmap=image_cmap)

        # crop images in needed
        if crop_x is not None:
            ax.set_xlim(crop_x[0], crop_x[1])
        if crop_y is not None:
            ax.set_ylim(crop_y[0], crop_y[1])

        if axis_y_flipped:
            ax.invert_yaxis()

        if plt_axis == "off":
            for spine in ax.spines.values():
                spine.set_visible(False)
            ax.tick_params(bottom=False, labelbottom=False, left=False, labelleft=False)

        counts = value_df.values.copy() # lwm
        print(counts) # lwm

        # plot spots as circles
        print(counts.shape[1]) #lwm#
        c_ord = list(np.arange(0, counts.shape[1]))
        # print("c_ord: ",c_ord) # lwm ##

        colors = np.zeros((*counts.shape, 4))
        weights = np.zeros(counts.shape)
        
        for c in c_ord:
            #print("c: ",c) # lwm
            if min_intensity is not None:
                min_color_intensity = min_intensity
            else:
                print(c,"min color intesity is None ! ")
                min_color_intensity = counts[:, c].min()  
            if max_intensity is not None:
                max_color_intensity = max_intensity 
            else:
                print(c,"max color intensity is None ! ") 
                max_color_intensity = np.min([np.quantile(counts[:, c], max_color_quantile), max_col[c]]) 
            print("min_color_intensity : ",min_color_intensity)
            print("max_color_intensity : ",max_color_intensity)
        
            rgb_function = get_rgb_function(cmap=cmaps[c], min_value=min_color_intensity, max_value=max_color_intensity)
           
            color = rgb_function(counts[:, c])
            color[:, 3] = color[:, 3] * alpha_scaling
            
            # 2024 - 1 - 10 #  lwm  
            norm = mpl.colors.Normalize(vmin=min_color_intensity, vmax=max_color_intensity) 
            #print(norm)
                
            if colorbar_position is not None:
                cbar_ticks = [
                    min_color_intensity,
                    np.mean([min_color_intensity, max_color_intensity]),
                    max_color_intensity,
                ]
                cbar_ticks = np.array(cbar_ticks)

                if max_color_intensity > 13:
                    cbar_ticks = cbar_ticks.astype(np.int32)
                else:
                    cbar_ticks = cbar_ticks.round(2)
          
                
                cbar = fig.colorbar(
                    mpl.cm.ScalarMappable(norm=norm, cmap= cmaps[c]), 
                    cax=cbar_axes[c],
                    orientation="horizontal",
                    extend="both",
                    ticks=cbar_ticks,
                ) 

                cbar.ax.tick_params(labelsize=colorbar_tick_size)
                max_color = rgb_function(max_color_intensity / 1.5)
                cbar.ax.set_title(labels[c], **{**{"size": 20, "color": max_color, "alpha": 1}, **colorbar_label_kw})

            colors[:, c] = color
            weights[:, c] = np.clip(counts[:, c] / (max_color_intensity + 1e-10), 0, 1)
            weights[:, c][counts[:, c] < min_color_intensity] = 0

        colors_ryb = np.zeros((*weights.shape, 3))

        for i in range(colors.shape[0]):
            colors_ryb[i] = rgb_to_ryb(colors[i, :, :3])

        def kernel(w):
            return w**2
        
        # gene expression color spot # 
        kernel_weights = kernel(weights[:, :, np.newaxis])

        weighted_colors_ryb = (colors_ryb * kernel_weights).sum(axis=1) / kernel_weights.sum(axis=1)

        weighted_colors = np.zeros((weights.shape[0], 4))

        weighted_colors[:, :3] = ryb_to_rgb(weighted_colors_ryb)

        weighted_colors[:, 3] = colors[:, :, 3].max(axis=1)
        
        ax.scatter(x=coords[:, 0], y=coords[:, 1], c=weighted_colors, s=circle_diameter**2)
        
        # add text
        if text is not None:
            bbox_props = dict(boxstyle="round", ec="0.5", alpha=text_box_alpha, fc="w")
            texts = []
            for x, y, s in zip(
                np.array(text.iloc[:, 0].values).flatten(),
                np.array(text.iloc[:, 1].values).flatten(),
                text.iloc[:, 2].tolist(),
            ):
                texts.append(ax.text(x, y, s, ha="center", va="bottom", bbox=bbox_props))

            if adjust_text:
                from adjustText import adjust_text

                adjust_text(texts, arrowprops=dict(arrowstyle="->", color="w", lw=0.5))

    return fig,ax

In [ ]:
def add_ScaleRuler(fig,x,y,length,size,label,lw,local,background):
    r'''
    :para fig: main plot 
    :para x: x axis position of scale ruler
    :para y: y axis position of scale ruler 
    :para length: length of scale ruler
    :para size:  y axis high of scale ruler
    :para label: unit of scale ruler ,Default : "10μm" 
    :para lw: line width,Default : 2
    '''
    if background == "dark_background":
        color = 'white'
    elif background == "fast":
        color = 'black'
    ax = fig.get_axes()

    ax[0].hlines(y= y,  xmin = x, xmax = x+length, colors="yellow", ls="-", lw=lw)
 
    return(fig)

In [ ]:
def CreateParaDict(Filelist,Samplelist,Genelist,NoneGenelist,AllGene,
                   Celltypes,SpotSize,Colorlist,Output):
    r'''
    :parameter Filelist : list
    :parameter Samplelist : list
    :parameter NoneGenelist : list
    :parameter Genelist : list 
    :parameter AllGene : list
    :parameter Celltypes : list
    :parameter SpotSize : list
    :parameter Colorlist: list 
    :parameter Output : str
    ...
    '''
    ParaDict = {'filelist' : Filelist,
                'samplelist': Samplelist,
                'NoneGenelist': NoneGenelist,
                'Genelist': Genelist,
                'AllGene': AllGene, 
                'Celltypes': Celltypes,
                'SpotSize': SpotSize,
                'Colorlist': Colorlist,
                'Output': Output}
    return(ParaDict)
datetime.datetime.now().strftime('%c')

In [ ]:
def MutilGeneExperssion(ParaDict,min_intensity,max_intensity,Figsize = (8,4),
                        Style = 'dark_background',ColorBarSize = 6,ColorbarTickSize = 6,size=60,
                        ruler_length = 400,label = "200 μm",lw = 4,x_shift = 500,
                        y_local = 600,method = "single"):
    r'''
    :parameter ParaDict : dict
    :parameter min_intensity : 
    :parameter max_intensity : 
    :parameter Figsize : tuple
    :parameter CircleDiameter : number 
    :parameter ColorBarSize : number
    :parameter Style : plot style (matplolib.style.context):
                          'fast' - white background & dark text;
                          'dark_background' - black background & white text;
    :parameter ColorbarTickSize : 
    :parameter size : 
    :parameter ruler.length : 
    :parameter label : 
    :parameter lw : 
    :parameter x.shift :
    :parameter y.local : 
    :parameter method : 
    '''
 
    # parameter #
    filelist = ParaDict['filelist']
    samplelist = ParaDict['samplelist']
    NoneGenelist = ParaDict['NoneGenelist']
    Genelist = ParaDict['Genelist']
    AllGene = ParaDict['AllGene']
    Celltypes = ParaDict['Celltypes']
    Colorlist = ParaDict['Colorlist']
    SpotSize = ParaDict['SpotSize']
    Output = ParaDict['Output']
    print(SpotSize)
    
    for i in range(0,len(samplelist)):
        file = filelist[i]
        print(file) # h5ad
        sample = samplelist[i]
        print("sample: ",sample) # sample
        out = output + "/" + sample
        print(out)
        if(os.path.exists(out)):
            print("Project Exist!")
        else :
            os.mkdir(out) # out project
        
        # input #
        adata = sc.read_h5ad(file)
        sc.pp.normalize_total(adata, target_sum=1e4) # normalize #
        sc.pp.log1p(adata) # log1p #
        print(adata)
        
        # adata.var_names : gene change number !
        # adata.var.index = adata.var["_index"] 
        adata.var.index = adata.var['features']
        GeneExp  = adata.to_df().copy()
        print(adata.var_names)
        
        # change y axis #
        adata.obs[['y']] = - adata.obs[["y"]] # notice # 
        adata.obsm['spatial'] = adata.obs[["x","y"]].values
        coord = adata.obsm["spatial"]
        
        # Get All Experssion Gene #
        NoneGene = NoneGenelist[i] # NoneGene 
        print("NoneGene: ",NoneGene)
        gene = list(set(AllGene).difference(NoneGene)) # All Experssion Gene 
        print(gene)
        gene2 = pd.DataFrame(gene).drop_duplicates()[0].to_list() # duplicate #
        print(gene2)
        
        # Gene Experssion data.frame #
        GeneExp = GeneExp[gene2]
        
        # celltypes #
        # genelist #
        
        if method == "single": # score or single expression #
            print("single")
            for j in range(0,len(Genelist)):
                print(j)
                ct = Celltypes[j] # each celltypes 
                gene3 = Genelist[j]
                genel = list(set(gene3).difference(NoneGene)) # Get each celltypes Experssion Gene 
                CircleDiameter = SpotSize[j]
                print("CircleDiameter: ",CircleDiameter)
                color = Colorlist[j]
                print("number of gene list :",len(genel))
                if len(genel) > 1 : # cal gene exp score 
                    print("genel: ",genel)
                    score_str = ct + "_scroe"
                    sc.tl.score_genes(adata,gene_list = genel,
                                    score_name = score_str,use_raw = False) # warnning # 
                    marker = [score_str]
                    GeneExp2 = adata.obs[score_str].to_frame() # only one gene list score
                    print("min score : ", np.min(GeneExp2[score_str]))
                    print("max score : ", np.max(GeneExp2[score_str]))
                    
                else :
                    marker = genel
                    GeneExp2 = GeneExp[genel]
                    
                print("marker: ",marker)
                print("GeneExp: ",GeneExp2)
                
                plt.rcParams["figure.figsize"] = Figsize # 
                fig, ax = plot_spatial_general1(value_df=GeneExp2, coords=adata.obsm["spatial"],
                                min_intensity = min_intensity,
                                max_intensity = max_intensity,
                                labels=marker, circle_diameter= CircleDiameter, 
                                style = Style, reorder_cmap = color,
                                colorbar_position="right", 
                                colorbar_tick_size=ColorbarTickSize, 
                                colorbar_label_kw={"size":ColorBarSize}, max_color_quantile=1, img_alpha=1) # 1
                ax.scatter(coord[:,0], coord[:,1], color="#E0E0E0", marker=".", zorder=0, s=0.1)
                fig = add_ScaleRuler(fig = fig,x=adata.obs["x"].max() + x_shift,y=adata.obs["y"].max(),
                                      length = ruler_length,size=size,label=label,lw=lw,background = Style,
                                      local = y_local)
                
        elif method == 'multiple': # multiple Gene Expression ##
            print("multiple")
            ct = Celltypes
            print("celltype: ",ct)
            CircleDiameter = SpotSize
            print("CircleDiameter:",CircleDiameter)
            color = Colorlist
            print("color",Colorlist)
            print("Genelsit",Genelist)
            genel = list(set(Genelist).difference(NoneGene))
            marker = genel
            GeneExp2 = GeneExp[genel]
            print("marker: ",marker)
            print(GeneExp2)
            plt.rcParams["figure.figsize"] = Figsize # 
            
            fig, ax = plot_spatial_general1(value_df=GeneExp2, coords=adata.obsm["spatial"], 
                                min_intensity = min_intensity,
                                max_intensity = max_intensity,
                                labels=marker, circle_diameter= CircleDiameter, 
                                style = Style, reorder_cmap = color,
                                colorbar_position="right", 
                                colorbar_tick_size=ColorbarTickSize,  
                                colorbar_label_kw={"size":ColorBarSize}, max_color_quantile=1, img_alpha=1) 
            ax.scatter(coord[:,0], coord[:,1], color="#E0E0E0", marker=".", zorder=0, s=0.1)
             # spleen -1000 # Default + 500 # SPF_thymus + 2000
            fig = add_ScaleRuler(fig = fig,x=adata.obs["x"].max() + x_shift ,y=adata.obs["y"].max(),
                                      length = ruler_length,size = size,label=label,background = Style,
                                      lw=lw,local = y_local)
            
            
        else : 
            print("Please check your method!") 
        
        output_fig = out + '/' + sample + "_" + ct + "_gene.pdf"
        print(output_fig)
        fig.savefig(output_fig,dpi = 512)

        print("--- end ----")

In [ ]:
def get_rgb_function(cmap, min_value, max_value):
    r"""Generate a function to map continous values to RGB values using colormap between min_value & max_value."""

    if min_value > max_value:
        raise ValueError("Max_value should be greater or than min_value.")

    if min_value == max_value:
        warnings.warn(
            "Max_color is equal to min_color. It might be because of the data or bad parameter choice. "
            "If you are using plot_contours function try increasing max_color_quantile parameter and"
            "removing cell types with all zero values."
        )

        def func_equal(x):
            factor = 0 if max_value == 0 else 0.5
            return cmap(np.ones_like(x) * factor)

        return func_equal

    def func(x):
        return cmap((np.clip(x, min_value, max_value) - min_value) / (max_value - min_value))

    return func


def rgb_to_ryb(rgb):
    """
    Converts colours from RGB colorspace to RYB

    Parameters
    ----------

    rgb
        numpy array Nx3

    Returns
    -------
    Numpy array Nx3
    """
    rgb = np.array(rgb)
    if len(rgb.shape) == 1:
        rgb = rgb[np.newaxis, :]

    white = rgb.min(axis=1)
    black = (1 - rgb).min(axis=1)
    rgb = rgb - white[:, np.newaxis]

    yellow = rgb[:, :2].min(axis=1)
    ryb = np.zeros_like(rgb)
    ryb[:, 0] = rgb[:, 0] - yellow
    ryb[:, 1] = (yellow + rgb[:, 1]) / 2
    ryb[:, 2] = (rgb[:, 2] + rgb[:, 1] - yellow) / 2

    mask = ~(ryb == 0).all(axis=1)
    if mask.any():
        norm = ryb[mask].max(axis=1) / rgb[mask].max(axis=1)
        ryb[mask] = ryb[mask] / norm[:, np.newaxis]

    return ryb + black[:, np.newaxis]


def ryb_to_rgb(ryb):
    """
    Converts colours from RYB colorspace to RGB

    Parameters
    ----------

    ryb
        numpy array Nx3

    Returns
    -------
    Numpy array Nx3
    """
    ryb = np.array(ryb)
    if len(ryb.shape) == 1:
        ryb = ryb[np.newaxis, :]

    black = ryb.min(axis=1)
    white = (1 - ryb).min(axis=1)
    ryb = ryb - black[:, np.newaxis]

    green = ryb[:, 1:].min(axis=1)
    rgb = np.zeros_like(ryb)
    rgb[:, 0] = ryb[:, 0] + ryb[:, 1] - green
    rgb[:, 1] = green + ryb[:, 1]
    rgb[:, 2] = (ryb[:, 2] - green) * 2

    mask = ~(ryb == 0).all(axis=1)
    if mask.any():
        norm = rgb[mask].max(axis=1) / ryb[mask].max(axis=1)
        rgb[mask] = rgb[mask] / norm[:, np.newaxis]

    return rgb + white[:, np.newaxis]


def plot_spatial_general(
    value_df,
    coords,
    labels,
    text=None,
    circle_diameter=4.0,
    alpha_scaling=1.0,
    max_col=(np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf),
    max_color_quantile=0.98,
    show_img=True,
    img=None,
    img_alpha=1.0,
    adjust_text=False,
    plt_axis="off",
    axis_y_flipped=True,
    x_y_labels=("", ""),
    crop_x=None,
    crop_y=None,
    text_box_alpha=0.9,
    reorder_cmap=range(7),
    style="fast",
    colorbar_position="bottom",
    colorbar_label_kw={},
    colorbar_shape={},
    colorbar_tick_size=12,
    colorbar_grid=None,
    image_cmap="Greys_r",
    white_spacing=40,
    max_value=None
):
    r"""Plot spatial abundance of cell types (regulatory programmes) with colour gradient and interpolation.

      This method supports only 7 cell types with these colours (in order, which can be changed using reorder_cmap).
      'yellow' 'orange' 'blue' 'green' 'purple' 'grey' 'white'

    :param value_df: pd.DataFrame - with cell abundance or other features (only 7 allowed, columns) across locations (rows)
    :param coords: np.ndarray - x and y coordinates (in columns) to be used for ploting spots
    :param text: pd.DataFrame - with x, y coordinates, text to be printed
    :param circle_diameter: diameter of circles
    :param labels: list of strings, labels of cell types
    :param alpha_scaling: adjust color alpha
    :param max_col: crops the colorscale maximum value for each column in value_df.
    :param max_color_quantile: crops the colorscale at x quantile of the data.
    :param show_img: show image?
    :param img: numpy array representing a tissue image.
        If not provided a black background image is used.
    :param img_alpha: transparency of the image
    :param lim: x and y max limits on the plot. Minimum is always set to 0, if `lim` is None maximum
        is set to image height and width. If 'no_limit' then no limit is set.
    :param adjust_text: move text label to prevent overlap
    :param plt_axis: show axes?
    :param axis_y_flipped: flip y axis to match coordinates of the plotted image
    :param reorder_cmap: reorder colors to make sure you get the right color for each category

    :param style: plot style (matplolib.style.context):
        'fast' - white background & dark text;
        'dark_background' - black background & white text;

    :param colorbar_position: 'bottom', 'right' or None
    :param colorbar_label_kw: dict that will be forwarded to ax.set_label()
    :param colorbar_shape: dict {'vertical_gaps': 1.5, 'horizontal_gaps': 1.5,
                                    'width': 0.2, 'height': 0.2}, not obligatory to contain all params
    :param colorbar_tick_size: colorbar ticks label size
    :param colorbar_grid: tuple of colorbar grid (rows, columns)
    :param image_cmap: matplotlib colormap for grayscale image
    :param white_spacing: percent of colorbars to be hidden

    """

    if value_df.shape[1] > 7:
        raise ValueError("Maximum of 7 cell types / factors can be plotted at the moment")

    def create_colormap(R, G, B):
        spacing = int(white_spacing * 2.55)

        N = 255
        M = 3

        alphas = np.concatenate([[0] * spacing * M, np.linspace(0, 1.0, (N - spacing) * M)])

        vals = np.ones((N * M, 4))
        #         vals[:, 0] = np.linspace(1, R / 255, N * M)
        #         vals[:, 1] = np.linspace(1, G / 255, N * M)
        #         vals[:, 2] = np.linspace(1, B / 255, N * M)
        for i, color in enumerate([R, G, B]):
            vals[:, i] = color / 255
        vals[:, 3] = alphas

        return ListedColormap(vals)

    # Create linearly scaled colormaps
    YellowCM = create_colormap(240, 228, 66)  # #F0E442 ['#F0E442', '#D55E00', '#56B4E9',
    # '#009E73', '#5A14A5', '#C8C8C8', '#323232']
    RedCM = create_colormap(213, 94, 0)  # #D55E00
    BlueCM = create_colormap(86, 180, 233)  # #56B4E9
    GreenCM = create_colormap(0, 158, 115)  # #009E73
    GreyCM = create_colormap(200, 200, 200)  # #C8C8C8
    WhiteCM = create_colormap(50, 50, 50)  # #323232
    PurpleCM = create_colormap(90, 20, 165)  # #5A14A5

    cmaps = [YellowCM, RedCM, BlueCM, GreenCM, PurpleCM, GreyCM, WhiteCM]

    cmaps = [cmaps[i] for i in reorder_cmap]

    with mpl.style.context(style):

        fig = plt.figure()

        if colorbar_position == "right":

            if colorbar_grid is None:
                colorbar_grid = (len(labels), 1)

            shape = {"vertical_gaps": 1.5, "horizontal_gaps": 0, "width": 0.15, "height": 0.2}
            shape = {**shape, **colorbar_shape}

            gs = GridSpec(
                nrows=colorbar_grid[0] + 2,
                ncols=colorbar_grid[1] + 1,
                width_ratios=[1, *[shape["width"]] * colorbar_grid[1]],
                height_ratios=[1, *[shape["height"]] * colorbar_grid[0], 1],
                hspace=shape["vertical_gaps"],
                wspace=shape["horizontal_gaps"],
            )
            ax = fig.add_subplot(gs[:, 0], aspect="equal", rasterized=False)

        if colorbar_position == "bottom":
            if colorbar_grid is None:
                if len(labels) <= 3:
                    colorbar_grid = (1, len(labels))
                else:
                    n_rows = round(len(labels) / 3 + 0.5 - 1e-9)
                    colorbar_grid = (n_rows, 3)

            shape = {"vertical_gaps": 0.3, "horizontal_gaps": 0.6, "width": 0.2, "height": 0.035}
            shape = {**shape, **colorbar_shape}

            gs = GridSpec(
                nrows=colorbar_grid[0] + 1,
                ncols=colorbar_grid[1] + 2,
                width_ratios=[0.3, *[shape["width"]] * colorbar_grid[1], 0.3],
                height_ratios=[1, *[shape["height"]] * colorbar_grid[0]],
                hspace=shape["vertical_gaps"],
                wspace=shape["horizontal_gaps"],
            )

            ax = fig.add_subplot(gs[0, :], aspect="equal", rasterized=False)

        if colorbar_position is None:
            ax = fig.add_subplot(aspect="equal", rasterized=False)

        if colorbar_position is not None:
            cbar_axes = []
            for row in range(1, colorbar_grid[0] + 1):
                for column in range(1, colorbar_grid[1] + 1):
                    cbar_axes.append(fig.add_subplot(gs[row, column]))

            n_excess = colorbar_grid[0] * colorbar_grid[1] - len(labels)
            if n_excess > 0:
                for i in range(1, n_excess + 1):
                    cbar_axes[-i].set_visible(False)

        ax.set_xlabel(x_y_labels[0])
        ax.set_ylabel(x_y_labels[1])

        if img is not None and show_img:
            ax.imshow(img, aspect="equal", alpha=img_alpha, origin="lower", cmap=image_cmap)

        # crop images in needed
        if crop_x is not None:
            ax.set_xlim(crop_x[0], crop_x[1])
        if crop_y is not None:
            ax.set_ylim(crop_y[0], crop_y[1])

        if axis_y_flipped:
            ax.invert_yaxis()

        if plt_axis == "off":
            for spine in ax.spines.values():
                spine.set_visible(False)
            ax.tick_params(bottom=False, labelbottom=False, left=False, labelleft=False)

        counts = value_df.values.copy()

        # plot spots as circles
        c_ord = list(np.arange(0, counts.shape[1]))

        colors = np.zeros((*counts.shape, 4))
        weights = np.zeros(counts.shape)

        for c in c_ord:

            min_color_intensity = counts[:, c].min()
            #max_color_intensity = np.min([np.quantile(counts[:, c], max_color_quantile), max_col[c]])
            max_color_intensity = max_col[c]

            rgb_function = get_rgb_function(cmap=cmaps[c], min_value=min_color_intensity, max_value=max_color_intensity)

            color = rgb_function(counts[:, c])
            color[:, 3] = color[:, 3] * alpha_scaling

            norm = mpl.colors.Normalize(vmin=min_color_intensity, vmax=max_color_intensity)

            if colorbar_position is not None:

                cbar_ticks = [
                    min_color_intensity,
                    np.mean([min_color_intensity, max_color_intensity]),
                    max_color_intensity,
                ]
                """
                step = (max_color_intensity - min_color_intensity) / 2
                cbar_ticks = np.arange(min_color_intensity, max_color_intensity, step)
                """
                cbar_ticks = np.array(cbar_ticks)

                if max_color_intensity > 13:
                    cbar_ticks = cbar_ticks.astype(np.int32)
                else:
                    cbar_ticks = cbar_ticks.round(2)

                print(cbar_ticks)
                cbar = fig.colorbar(
                    mpl.cm.ScalarMappable(norm=norm, cmap=cmaps[c]),
                    cax=cbar_axes[c],
                    orientation="horizontal",
                    extend="neither",
                    ticks=cbar_ticks
                )
                cbar.ax.tick_params(labelsize=colorbar_tick_size)
                max_color = rgb_function(max_color_intensity / 1.5)
                cbar.ax.set_title(labels[c], **{**{"size": 20, "color": max_color, "alpha": 1}, **colorbar_label_kw})

            colors[:, c] = color
            weights[:, c] = np.clip(counts[:, c] / (max_color_intensity + 1e-10), 0, 1)
            weights[:, c][counts[:, c] < min_color_intensity] = 0

        colors_ryb = np.zeros((*weights.shape, 3))

        for i in range(colors.shape[0]):
            colors_ryb[i] = rgb_to_ryb(colors[i, :, :3])

        def kernel(w):
            return w**2

        kernel_weights = kernel(weights[:, :, np.newaxis])
        weighted_colors_ryb = (colors_ryb * kernel_weights).sum(axis=1) / kernel_weights.sum(axis=1)

        weighted_colors = np.zeros((weights.shape[0], 4))

        weighted_colors[:, :3] = ryb_to_rgb(weighted_colors_ryb)

        weighted_colors[:, 3] = colors[:, :, 3].max(axis=1)

        ax.scatter(x=coords[:, 0], y=coords[:, 1], c=weighted_colors, s=circle_diameter**2)

        # add text
        if text is not None:
            bbox_props = dict(boxstyle="round", ec="0.5", alpha=text_box_alpha, fc="w")
            texts = []
            for x, y, s in zip(
                np.array(text.iloc[:, 0].values).flatten(),
                np.array(text.iloc[:, 1].values).flatten(),
                text.iloc[:, 2].tolist(),
            ):
                texts.append(ax.text(x, y, s, ha="center", va="bottom", bbox=bbox_props))

            if adjust_text:
                from adjustText import adjust_text

                adjust_text(texts, arrowprops=dict(arrowstyle="->", color="w", lw=0.5))

    return fig, ax


def plot_spatial(adata, color, img_key="hires", show_img=True, **kwargs):
    """Plot spatial abundance of cell types (regulatory programmes) with colour gradient
    and interpolation (from Visium anndata).

    This method supports only 7 cell types with these colours (in order, which can be changed using reorder_cmap).
    'yellow' 'orange' 'blue' 'green' 'purple' 'grey' 'white'

    :param adata: adata object with spatial coordinates in adata.obsm['spatial']
    :param color: list of adata.obs column names to be plotted
    :param kwargs: arguments to plot_spatial_general
    :return: matplotlib figure
    """

    if show_img is True:
        kwargs["show_img"] = True
        kwargs["img"] = list(adata.uns["spatial"].values())[0]["images"][img_key]

    # location coordinates
    if "spatial" in adata.uns.keys():
        kwargs["coords"] = (
            adata.obsm["spatial"] * list(adata.uns["spatial"].values())[0]["scalefactors"][f"tissue_{img_key}_scalef"]
        )
    else:
        kwargs["coords"] = adata.obsm["spatial"]

    fig, ax = plot_spatial_general(value_df=adata.obs[color], **kwargs)  # cell abundance values

    return fig, ax


In [ ]:
fig, ax = plot_spatial(adata=SPF_bin20[SPF_bin20.obs["Plasma1"]!=0], color=["Plasma1"], show_img=False, labels=["Plasma"], 
                          circle_diameter=0.5, alpha_scaling=1.0, max_col=(0.4034, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf),
                            max_color_quantile=1,  img=None, img_alpha=0, adjust_text=True, plt_axis="off",
                            axis_y_flipped=True, x_y_labels=("", ""), text_box_alpha=0.9, reorder_cmap=[0], style="fast",
                            colorbar_position="right", colorbar_label_kw={"size":10}, colorbar_shape={}, colorbar_tick_size=6,
                            colorbar_grid=[1,1], white_spacing=10)#, image_cmap="Greys_r"

def shapely_to_paths(g):
    if isinstance(g, MultiLineString):
        return [list(ls.coords) for ls in g.geoms]
    else:
        raise Exception('unhandled shapely geometry: %s' % type(g))



#获取路径
paths = shapely_to_paths(SPF_points_res.boundary)
# 遍历路径并添加到图表中
for path in paths:
    # 解包路径中的坐标
    x, y = zip(*path)
    # 绘制线条
    ax.plot(x, y, color="cyan", linewidth=0.4, alpha=1)

paths_l9 = shapely_to_paths(SPF_points_l9_res.boundary)
# 遍历路径并添加到图表中
for path in paths_l9:
    # 解包路径中的坐标
    x, y = zip(*path)
    # 绘制线条
    ax.plot(x, y, color="#FF00FF", linewidth=0.4, alpha=1)

point0 = [20000, 20000]
point1 = [22000, 20000]
ax.plot([point0[0], point1[0]], [point0[1], point1[1]], color='yellow', linestyle="-")
plt.savefig("SPF.Plasma.l1l9.test.pdf")

# Fig 2 

In [ ]:
dotplot_custom <- function(df_nums,rowlist,collist,col_names){
    df_nums$celltypes <- rownames(df_nums)
    range01 <- function(x){(x - min(x)) / (max(x) - min(x))}
    df_prop <-  df_nums %>% select(-c("celltypes"))
    df_prop <- apply(df_prop, 2, function(x){x/sum(x)*100})
    rewrite0_row <- which(apply(df_prop,1,min)!=0)
    df_prop <- as.data.frame(t(apply(df_prop, 1, range01)))
    minpseudo <- min(df_prop[df_prop!=0])/2
    for (i in rewrite0_row){
      df_prop[i,which(df_prop[i,]==0)] <- minpseudo
    }
    df_prop$celltypes <- df_nums$celltypes
    
    df_prop_long <- reshape2::melt(df_prop, id.var='celltypes', variable.name='Organs', value.name = "Propstat")
    df_nums_long <- reshape2::melt(df_nums, id.var='celltypes', variable.name='Organs', value.name = "Numstat")

    df_long <- dplyr::left_join(df_prop_long,df_nums_long,by=c("celltypes","Organs"))
    df_long$celltypes <- as.factor(df_long$celltypes)
    df_long$Organs <- as.factor(df_long$Organs)
    df_long$Numstat[df_long$Numstat==0] <- NA 
    df_long$Propstat[df_long$Propstat==0] <- NA
    
    df_newname <- data.frame(celltypes=rowlist, newname= rowlist)
    
    df_long_newname <- merge(df_long,df_newname,by="celltypes",all.x = T)
    
    df_long <- df_long_newname[,c(5,2,3,4)]
    colnames(df_long)[1] <- "celltypes"
    print(head(df_long)) 
    print(unique(df_long$celltypes)) 
    df_long$celltypes <- factor(df_long$celltypes, levels = rev(df_newname$newname))
    df_long$Organs <- factor(df_long$Organs, levels = c(collist))
    df_long$Numstat <- cut(df_long$Numstat, breaks = c(-Inf, 100, 250, 500, 750, 1000, 5000, Inf), 
                           labels = c("1-100", "101-250", "251-500", "501-750", "750-1000", "1000-5000", ">5000"), right=FALSE)

    p <- ggplot(df_long,aes(x=Organs,y=celltypes))+
              geom_point(aes(size=Propstat, color=Numstat))+
              theme_bw()+
                guides(shape = guide_legend(override.aes = list(size=20)))+
          theme(axis.text.x = element_text(size = 15, face="bold", vjust = 0.5, hjust = 0.5, angle = 0),
                axis.text.y = element_text(size = 15, face="bold"))+
          theme(legend.title = element_text(size = 15, face="bold"),
                legend.text = element_text(size = 13)) +
          scale_color_manual(values = col_names[3:10]) +  labs(x=NULL,y=NULL)  + coord_flip() 
    return(p)
}
GetMergeTable <- function(Object,var){
    print(Object)
    whole <- as.data.frame.array(table(Object@meta.data$clusters,Object@meta.data$mice))
    print(whole)
    Idents(Object) <- var
    for( i in unique(Object@meta.data[,var])){
        print(i)
        subObj <- subset(Object, idents = i)
        tissue_table <- as.data.frame.array(table(subObj@meta.data$clusters,subObj@meta.data$mice))
        tissue_prefix <- rep(i,2)
        table_newnames <- paste(colnames(tissue_table),tissue_prefix,sep = "_")
        colnames(tissue_table) <- table_newnames
        whole <- cbind(whole,tissue_table)
    }
    return(whole)
}
date()

# Fig 3 

In [ ]:
DoEechPiePlot <- function(Counts,Genelist){## Experssion ##
 # Counts <- Counts[,Genelist]
  #print(head(Counts))
  data <- as.data.frame(Matrix::colSums(x = Counts > 0)) # whole express gene Connts > 0 # 
  colnames(data) <- "ExpStat"
  data$whole <- dim(Counts)[1] #  all spot ##
  data$gene <- rownames(data)
  data <- data %>% mutate(UnExpStat = whole - ExpStat)
  data$whole <- NULL
  plotlist <- list()
  for(i in 1:length(Genelist)){
    print(i)
    marker <- Genelist[i]
    print(marker)
    data.marker <- data %>% filter(gene == marker)
    print(data.marker)
    plot.marker <- melt(data.marker,id.vars = "gene")
    print(plot.marker)
    plot.marker$variable <- factor(plot.marker$variable,levels = c("UnExpStat","ExpStat"))
    plotlist[[marker]] <-  ggplot(data = plot.marker, mapping = aes(x = gene, y = value, fill = variable)) + 
                                geom_bar(stat = 'identity',  position = 'stack')+ coord_polar(theta = 'y') +
                                scale_fill_manual(values=c("#808080","#bd0000")) + 
                                labs(x = '', y = '', title = marker) + theme_void() + theme(
                                      plot.title = element_text(size = 25,family = "Times",face = "bold",hjust = 0.5),
                                legend.position = "none") 
  }
  return(plotlist)
}
DoUnionPiePlot <- function(Counts,Genelist,celltype){ 
  Counts <- Counts[,Genelist] ### question ###
  data <- as.data.frame(Counts > 0)
  data$Spot <- rownames(data)
  print(head(data))
  union.gene <- c()
  for(j in Genelist){
    print(j)
    exp.gene <- data[data[,j],"Spot"] # stat express Gene # 
    print(head(exp.gene))
    union.gene <- union(union.gene,exp.gene) # union Gene # 
  }
  print(length(union.gene))
  data.score  <- data.frame(ExpStat = length(union.gene),
                            UnExpStat = dim(Counts)[1] - length(union.gene),
                            celltype = celltype)
  print(data.score)
  plot.score <- melt(data.score,id.vars = "celltype")
  plot.score $variable <- factor(plot.score$variable,levels = c("UnExpStat","ExpStat"))
  p <- ggplot(data = plot.score, mapping = aes(x = celltype, y = value, fill = variable)) + 
    geom_bar(stat = 'identity',  position = 'stack')+ coord_polar(theta = 'y') +
    scale_fill_manual(values=c("#808080","#bd0000")) + 
    labs(x = '', y = '', title = celltype) + theme_void() + theme(
      plot.title = element_text(size = 16,family = "Times",face = "bold",hjust = 0.5),
      legend.position = "none") 
  return(p)
}

CellTypeMarkerPiePlot <- function(Counts,Celltypes,Markerlist,mice,sample,output,method){
  if(length(Celltypes) != length(Markerlist)){
    stop("Please check your number of vector of celltypes or marker ! ")
  }
  for(i in 1:length(Celltypes)){
    print(i)
    ct <- Celltypes[i]
    print(ct)
    mk <- Markerlist[[i]]
    print(mk)
    if(method == "experssion"){
      # each celltype Pieplot #
      pl <- DoEechPiePlot(Counts = Counts,Genelist = mk)
      p <- wrap_plots(pl,ncol = 1)
    } else if (method == "score"){
      p <- DoUnionPiePlot(Counts = Counts,Genelist = mk, celltype = ct)
      mk <- 1
    } else {
      stop("Please check your method !")
    } 
    # save #
    outfile <- paste(output,"/",sample,"_",ct,"_",method,".pdf",sep = "")
    print(outfile)
    pdf(outfile,w = 2,h = 2*length(mk))
    print(p)
    dev.off()
    print("--- end --- ")
  }
}
date()

# Fig 4 

In [ ]:
Cell2ClusterHeatmap <- function(data,hm_columns,tree_columns,ax.tx.size =20,ax.tl.size=25){
  # hm_dat #
  hm_dat <- data[,hm_columns]b
  hm_dat <- gather(hm_dat,celltypes,scores,-c("region_cluster"))
  hm_dat$scores <- log2(hm_dat$scores) # log10 == log2 yes!
  hm_dat$scores <-as.vector(scale(hm_dat$scores, center = T, scale = T))
  # tree_dat #
  tree_dat<-spread(hm_dat,celltypes,scores)
  rownames(tree_dat) <- tree_dat$region_cluster
  tree_dat <- tree_dat[,tree_columns]
  # check #
  print(head(hm_dat))
  str(hm_dat)
  print(head(tree_dat))
  print(length(colnames(tree_dat)))
  print(max(hm_dat$scores))
  print(min(hm_dat$scores))
  # heatmap #
  p <- ggplot(hm_dat,aes(x = celltypes,y = region_cluster,fill = scores)) + geom_tile() +# geom_raster() + 
    theme(axis.text.x = element_text(angle = 45,vjust = 1,hjust = 1,size = ax.tx.size,family = "Times",face = "bold", 
                                     margin = margin(5,0,0,0)),
          axis.text.y = element_text(size = ax.tx.size,family = "Times",face = "bold", 
                                     margin = margin(0,5,0,0)),
          axis.title= element_blank(),
          axis.line = element_line(linetype = 1,color= "black",size = 1),
          legend.text = element_text(size = 20,family = "Times",face = "bold"),
          legend.title = element_text(size = 25,family = "Times",face = "bold"),
          axis.ticks  = element_line(color = "black",size = 1,lineend = 2))  +
    scale_fill_gradient2(high =  "red",mid = "white", low = "blue",midpoint = 0) + labs(fill = "log2(Scores)")
  # tree plot #
  p2 <- ggtree(hclust(dist(tree_dat))) + labs(y = "region_cluster") + 
          theme(axis.title.y= element_text(size = ax.tl.size,family = "Times",face = "bold",margin = margin(0,5,0,0)))
  p3 <- ggtree(hclust(dist(t(tree_dat)))) + labs(y = "Celltypes") + 
    theme(axis.title.x= element_text(size = ax.tl.size,family = "Times",face = "bold",margin = margin(5,0,0,0))) + 
    coord_flip()
  # joint #
  p.me <- p %>% insert_left(p2,width = 0.1) %>% insert_bottom(p3,height = 0.4)  
  return(p.me)
}

In [ ]:
GetCell2ClusterAvgScore <- function(dat,hm_index = c(13:66,69)){
  hm_dat <- dat[,hm_index]
  mat_index <- seq(1:(length(colnames(hm_dat))-1))
  avg_score <- aggregate(hm_dat[,mat_index],by = list(region_cluster = hm_dat$region_cluster),FUN = "mean")
  str(avg_score)
  rownames(avg_score) <- avg_score$region_cluster
  avg_score <- avg_score[,-1]
  str(avg_score)
  score <- apply(avg_score,MARGIN = 2,FUN = function(x){
    x / sum(x)
  })
  score <- as.data.frame(score)
  score$region_cluster <- rownames(score)
  return(score)
}

In [ ]:
# function data# 
getdf <- function(obj){
  aobj <- NULL
  obj <- readRDS(obj)
  aobj[[1]] <- obj@assays$METABOLISM$score
  aobj[[2]] <- obj@meta.data
  return(aobj)
}
# ct : c("EC(Car1 high)","EC(Hmgcs2 high)","Goblet1")
meltdf <- function(obj,ct){
    metabolism <- obj[[1]]
    metadata <- obj[[2]][,c("mice","clusters","celltypes")]
    df <- merge(metadata,t(metabolism),by="row.names")
    df_select <- df %>% filter(celltypes %in% ct)
    df_select$celltypes <- factor(df_select$celltypes, levels=ct)
    return(df_select)
}
getmedian <- function(target_profile){
  pathway_table <- NULL
  for (i in c(1:nlevels(target_profile$celltypes))){
    for (j in c("GF","SPF")){
      tmp <- subset(target_profile,celltypes == levels(target_profile$celltypes)[i] & mice==j)
      tmp <- tmp[,c(3:ncol(target_profile))]
      tmp_mean <- as.data.frame(t(apply(tmp,2,median)))
      rownames(tmp_mean) <- paste0(j,"_",levels(target_profile$celltypes)[i])
      pathway_table <- rbind(pathway_table,tmp_mean)
    }
  }
  return(pathway_table)
}
megedf <- function(obj.list,ctlist){
    range01 <- function(x, ...){(x - min(x, ...)) / (max(x, ...) - min(x, ...))}
    metalist <- list()
    for(i in names(obj.list)){
        print(i)
        obj <- obj.list[[i]]
        print(obj)
        ct <- ctlist[[i]]
        print(ct)
        cat("Running functon of getdf","\n")
        metaobj <- getdf(obj = obj)
        cat("Running functon of meltdf","\n")
        df_select <- meltdf(obj = metaobj,ct = ct)
        cat("Running functon of getmetdiandf","\n")
        df_median <- getmedian(df_select[,c(2,4:83)]) ## colon problem # 
        df_median[df_median < 0.25] <- 0
        metalist[[i]] <- df_median
    } 
    metabolism_18 <- bind_rows(metalist[["cecum"]], metalist[["colon"]],metalist[["ileum"]])

    metabolism_GF <- metabolism_18[seq(1,18, by=2),]
    metabolism_SPF <- metabolism_18[seq(2,18, by=2),]
    df_prop_18 <- as.data.frame(apply(metabolism_18, 2, range01))
    df_nums_18 <- metabolism_18

    df_prop_GF <- as.data.frame(apply(metabolism_GF, 2, range01))
    df_prop_GF[is.na(df_prop_GF)] <- 0
    df_nums_GF <- metabolism_GF
    df_prop_SPF <- as.data.frame(apply(metabolism_SPF, 2, range01))
    df_prop_SPF[is.na(df_prop_SPF)] <- 0
    df_nums_SPF <- metabolism_SPF

    df_prop_9_9 <- bind_rows(df_prop_GF, df_prop_SPF)
    df_nums_9_9 <- bind_rows(df_nums_GF, df_nums_SPF)

    df_nums_9_9_filter <- df_nums_9_9[,which(apply(df_nums_9_9,2,sum)>0)]
    df_prop_9_9_filter <- df_prop_9_9[,which(apply(df_nums_9_9,2,sum)>0)]
    pdata <- list(df_nums = df_nums_9_9_filter, df_prop = df_prop_9_9_filter)
    return(pdata)
}

getbubbleplot2 <- function(df_prop, df_nums, celltype_order){
  pbb <- NULL
  
  df_tree <- t(df_prop)
  res_clu <- hclust(dist(df_tree))
  tree_order <- ggtree(res_clu)$data %>% filter(isTip==TRUE) %>% arrange(desc(y)) %>% select(label) %>% c()
  
  df_prop <- df_prop %>% tibble::rownames_to_column(var = "celltypes")
  df_nums <- df_nums %>% tibble::rownames_to_column(var = "celltypes")
  
  df_prop_long <- reshape2::melt(df_prop, id.var='celltypes', variable.name='Pathways', value.name = "ScaledScore")
  df_nums_long <- reshape2::melt(df_nums, id.var='celltypes', variable.name='Pathways', value.name = "Score")
  df_long <- dplyr::left_join(df_prop_long,df_nums_long,by=c("celltypes","Pathways"))
  
  df_long$celltypes <- factor(df_long$celltypes, levels = celltype_order)
  df_long$Pathways <- factor(df_long$Pathways, levels = rev(tree_order$label))
  df_long <- df_long %>% arrange(celltypes)
  df_long[df_long==0] <- NA
  
  p <- ggplot(df_long,aes(x=Pathways,y=celltypes))+
    geom_point(aes(size=ScaledScore, color=ScaledScore))+
    theme_bw()+
    theme(
      axis.text.x = element_text(size = 20, face="bold", vjust = 1, hjust = 1, angle = 75),
      axis.text.y = element_text(size = 20, face="bold"))+
    theme(legend.title = element_text(size = 20, face="bold"),
          legend.text = element_text(size = 15),
          legend.position = "right",legend.justification = "right",plot.margin = margin(2,2,2,2,"cm"))+
          #legend.position = "bottom",legend.justification = "right")+ 
          #legend.position = "right") + 
    scale_color_gradientn(colours = wes_palette("Zissou1", 100, type = "continuous"))+
    # scale_color_gradient(low="lightgrey",high="blue")+
    labs(x=NULL,y=NULL) + coord_flip() +  scale_x_discrete(position = "top") 
  
  p4 <- ggplot() +
    geom_boxplot(data=df_long, mapping=aes(x=celltypes, y=ScaledScore, fill=celltypes)) + 
    theme_bw()+
        theme(axis.title.x = element_blank(),
          axis.title.y = element_blank(),
          axis.text.y = element_text(size = 20,face="bold"),
          axis.text.x = element_text(size = 20, face="bold",vjust = 1, hjust = 1,angle = 60)) + 
        theme(legend.position = "none",plot.margin = margin(2,2,2,2,"cm")) #+coord_flip()
  
  pbb[[1]] <- p
  pbb[[2]] <- p4
  return(pbb)
}

In [ ]:
# Fig 6 

In [ ]:
DensityPlot <- function(data,title,color_str=c('GF'="#0000ff",'SPF'="#ff0000"),
                        xlim=c(-5,2),x_breaks = 1,
                        ylim=c(0,0.7),y_breaks = 0.2,labels){
    data$group <- factor(data$group,levels = c("GF","SPF"))
    p <-ggplot(data, aes(log10(value+0.00001), fill = group,alpha=0.8)) +
        geom_density() +
      labs(title = title, x = "",y = "") + theme_classic() + 
      theme(plot.title = element_text(size = 25,hjust = 0.5,face = "bold"))+
      theme(axis.text = element_text(size = 20),axis.title = element_text(size = 20),
      legend.text = element_text(size = 20 ),
      legend.title = element_text(size = 20 ,face = "bold")) + 
      scale_x_continuous(limits = xlim, breaks = seq(xlim[1],xlim[2],x_breaks),
                      labels = labels,expand = c(0,0))+
      scale_y_continuous(limits = ylim,breaks = seq(ylim[1],ylim[2],y_breaks),
                          expand = c(0,0))+
      scale_fill_manual(values=color_str,labels=c('GF','SPF'))+
      guides(alpha="none")
    return(p)
}
fancy_scientific <- function(l) {
  # turn in to character string in scientific notation
  l <- format(l, scientific = TRUE)
  l <- gsub("0e\\+00","0",l)
  # quote the part before the exponent to keep all the digits
  l <- gsub("^(.*)e", "'\\1'e", l)
  # remove "+" after exponent, if exists. E.g.: (3x10^+2 -> 3x10^2) 
  l <- gsub("e\\+","e",l)
  # turn the 'e+' into plotmath format
  l <- gsub("e", "%*%10^", l)
  # convert 1x10^ or 1.000x10^ -> 10^ 
  l <- gsub("\\'1[\\.0]*\\'\\%\\*\\%", "", l)
  # return this as an expression
  parse(text=l)
}
tran_scientific <- function(x){
    s <- c()
    for(i in x){
      #print(i)
      f <- fancy_scientific(i)
      s <- c(s,f)
    }
    return(s)
}
MultiDenistyPlot <- function(dat_list,color_str=c('GF'="#0000ff",'SPF'="#ff0000"),
                        xlim=c(-5,2),x_breaks = 1,
                        ylim=c(0,0.7),y_breaks = 0.2,labels){
    plist <- list()
    for(i in names(dat_list)){
        plist[[i]] <- DensityPlot(data = dat_lst[[i]],title = i,color_str = color_str,
                                  xlim = xlim,ylim=ylim,x_breaks = x_breaks,
                                  y_breaks = y_breaks,labels = labels)
    }
    return(plist)
}

In [ ]:
CaluLinkList <- function(lst,sort_Abbreviation){
    dat_lst <- list()
    for(i in names(lst)){
        print(i)
        data <- lst[[i]]
        dat_lst[[i]]$counts <- data
        colnames(data)[2] <- "Group_type"
        dat <- data %>% select(Group_type,Abbreviation, value)
        dat <- spread(dat,key = Group_type,value = value)
        dat$Tissue <- i
        dat$Abbreviation <- as.character(dat$Abbreviation)
        print(head(dat,2))
        dat_lst[[i]]$link <- dat  %>% mutate(Abbreviation = fct_relevel(Abbreviation,sort_Abbreviation)) %>%
                    arrange(by=desc(Abbreviation)) %>%
                        mutate(GroupA=cumsum(.data[[colnames(dat)[2]]]), 
                               GroupB=cumsum(.data[[colnames(dat)[3]]]))
        print('--end--')
    }
    return(dat_lst)
}
WrapPlot <- function(lst,mycolor,ncol,ax.tx.size = 20,
                     lg.tx.size = 16,
                     lg.tl.size = 25){
    # theme self # 
    mytheme <- theme_classic() + theme(
        axis.text.y = element_text(size = ax.tx.size,face = "bold",margin = margin(0,5,0,0)),
        axis.text.x = element_text(size = ax.tx.size,face = "bold",margin = margin(5,0,0,0)),
        axis.line = element_line(linetype = 1,color= "black",size = 1),
        axis.ticks = element_line(linetype = 1,color= "black",size = 1,lineend = 100),
        legend.text = element_text(size = lg.tx.size,face = "bold"),
        legend.title = element_text(size = lg.tl.size,face = "bold")) + 
               theme(legend.key.size = unit(1,"cm"), 
                  legend.spacing.x = unit(0.2,'cm'),
                  legend.spacing.y = unit(0.2,'cm'),
                  legend.text = element_text(margin = margin(t = 0,r = 0,b = 0,l =  0))) + 
               theme(plot.title = element_text(size = 25,hjust = 0.5,face = "bold"),plot.margin = margin(0,0,0,0))
    plst <- lst()
    for(i in names(lst)){
        counts <- lst[[i]]$counts
        print(head(counts,2))
        links <- lst[[i]]$link
        print(head(links,2))
        counts$Abbreviation <- factor(counts$Abbreviation,levels = levels(links$Abbreviation))
        str(counts)
        plst[[i]] <-  ggplot(counts, aes(x=Group, y=value, fill=Abbreviation)) +
                geom_bar(stat = "identity", width=0.5, col='black')  +
                geom_segment(data=links, aes(x=1.25, xend=1.75, y=GroupA, yend=GroupB)) +
                scale_fill_manual(values=mycolor) + xlab('') +
                ylab(" ") + labs(title = i) + 
        guides(fill = guide_legend(ncol = 2, byrow = TRUE,override.aes = list(size = 10)))+ mytheme  
    }
    p <- wrap_plots(plst,ncol= ncol,guides = "collect")
    return(p)
}
WrapflowPlot <- function(lst,mycolor,order,ncol,ax.tx.size = 20,
                     lg.tx.size = 16,
                     lg.tl.size = 25){
    # theme self # 
    mytheme <- theme_classic() + theme(
        axis.text.y = element_text(size = ax.tx.size,face = "bold",margin = margin(0,5,0,0)),
        axis.text.x = element_text(size = ax.tx.size,face = "bold",margin = margin(5,0,0,0)),
        axis.line = element_line(linetype = 1,color= "black",size = 1),
        axis.ticks = element_line(linetype = 1,color= "black",size = 1,lineend = 100),
        legend.text = element_text(size = lg.tx.size,face = "bold"),
        legend.title = element_text(size = lg.tl.size,face = "bold")) + 
               theme(legend.key.size = unit(1,"cm"), 
                  legend.spacing.x = unit(0.2,'cm'),
                  legend.spacing.y = unit(0.2,'cm'),
                  legend.text = element_text(margin = margin(t = 0,r = 0,b = 0,l =  0))) + 
               theme(plot.title = element_text(size = 25,hjust = 0.5,face = "bold"),plot.margin = margin(0,0,0,0))
    plst <- lst()
    for(i in names(lst)){
        counts <- lst[[i]]
        counts$Abbreviation <- factor(counts$Abbreviation,levels = order)
        str(counts)
        plst[[i]] <-  ggplot(counts, aes(x = Group, y = value, fill = Abbreviation,
                                      stratum = Abbreviation, alluvium = Abbreviation )) +
                geom_col(width = 0.5,color= "black") + 
                geom_flow(width = 0.5,alpha = 0.4, knot.pos=0.5) + theme_classic()  +
                scale_fill_manual(values=mycolor) + xlab('') +
                ylab(" ") + labs(title = i) + 
        guides(fill = guide_legend(ncol = 2, byrow = TRUE,override.aes = list(size = 10)))+ mytheme  
    }
    p <- wrap_plots(plst,ncol= ncol,guides = "collect")
    return(p)
}
date()

In [ ]:
mulitBarplot <- function(dat,color=c("#955117","#F3E295")){
    dat$Group<-factor(dat$Group,levels=c("SPF","GF"))
    dat$variable<-factor(dat$variable,levels=c("mean_non_OH","mean_OH"))
    theme_self <- theme(plot.title = element_text(size = 25,hjust = 0.5,face = "bold"),
                        axis.text = element_text(size = 16),
                        #axis.title = element_text(size = 23),
                        legend.text = element_text(size = 16),
                        legend.title = element_text(size = 18 ,face = "bold"),
                        axis.title.y = element_text(size=18,margin = margin(0,10,0,0))) + 
                  theme(strip.background = element_blank(),strip.text =  element_text(size = 23,angle = 0))
    p <-ggplot(dat,aes(x=Group,y=Relative_Abundance,fill=variable))+ facet_wrap(~Type,ncol = 4) +
                  geom_bar(stat="identity",width = 0.7)+theme_classic()+
                  scale_fill_manual(values=color,labels = c("non 12-OH","12-OH"))+
                  labs(title = "", x = "", y = "Realtive Abundance")  + theme_self  +
                    guides(fill = guide_legend(title = "BA type",ncol = 1, byrow = TRUE))
    return(p)
}

In [ ]:
DoBoxplot <- function(data,x = "Group.x",y = "CDCA_CA" , color = c("#FF0000","#0000FF"),title){
    data$Group.x<-factor(data$Group.x, levels=c("GF","SPF"))
    theme_self <- theme(plot.title = element_text(size = 23,hjust = 0.5,face = "bold"),
                        axis.text = element_text(size = 15,colour="#000000"),
                        axis.title = element_text(size = 18,colour="#000000"),
                        legend.text = element_text(size = 15),
                      legend.title = element_text(size = 18 ,face = "bold"))
    p <- ggplot(data=data,aes_string(x = x,y = y))+
              geom_boxplot(aes(fill=Group.x),outlier.colour=NA,width=0.4,position="dodge",alpha=1)+
              scale_fill_manual(values=color,labels=c('GF','SPF'))+
              scale_x_discrete(limits = c("SPF","GF")) + 
              theme_classic()+labs(x="",y="",title =title)+
              geom_jitter(position=position_jitter(0.08))+
              geom_signif(comparisons=list(c('SPF',"GF")),
              test=wilcox.test,step_increase=0.1,textsize=5,
              map_signif_level=T, colour="#000000") + theme_self + guides(fill = guide_legend(title = "Group"))
    return(p)

}

In [ ]:
sort=['GF_Hepatocytes-Portal','GF_Hepatocytes-Mid','GF_Hepatocytes-Central',
      'SPF_Hepatocytes-Portal','SPF_Hepatocytes-Mid','SPF_Hepatocytes-Central']
liver.obs['celltypes'].cat.reorder_categories(sort,inplace=True)
sc.set_figure_params(scanpy=True, fontsize=24)
plt.rcParams["figure.figsize"] = (27,9) # 12
fig, ax = plt.subplots(1,1)
sc.pl.dotplot(liver, FG1, groupby="celltypes",ax = ax,standard_scale = "var", var_group_rotation=0 , use_raw = False , 
              dot_max = 0.7,show=False,linewidth=0,cmap = mycolor)